# 04 · 装饰器、上下文管理器与生成器 Decorators, Context Managers & Generators

> **能力标签**：SH1（Python 编程能力）

## 目标 Objectives

完成本课后，你应该能够：

1. 理解**函数是一等公民（first-class functions）**：函数可以作为参数传入、从函数返回、赋值给变量。
2. 理解**闭包（closure）**：内层函数"记住"外层函数的变量。
3. 写**装饰器（decorator）**：用 `functools.wraps` 保留被包装函数的元信息。
4. 理解 `@property` 装饰器：把方法变成只读属性（见第 02 课）。
5. 用 `@contextmanager` 实现**上下文管理器（context manager）**，理解 `with` 语句的执行流程。
6. 用 `yield` 写**生成器（generator）**，理解惰性求值（laziness）的好处。

> 对应能力：**SH1**


## 概念讲解 Concepts

### 一等公民函数（First-Class Functions）

在 Python 中，函数和整数、字符串一样，是可以传递的**对象**：

```python
def square(x):
    return x * x

f = square          # 赋值
print(f(5))         # 25

def apply(fn, x):   # 函数作为参数
    return fn(x)

apply(square, 5)    # 25
```

---

### 闭包（Closure）

内层函数可以"捕获"并记住外层函数的局部变量，即使外层函数已经返回：

```python
def make_adder(n):
    def adder(x):
        return x + n   # n 被"捕获"在闭包里
    return adder

add5 = make_adder(5)
add5(10)   # 15
```

装饰器的核心机制就是闭包。

---

### 装饰器（Decorator）

装饰器是一个函数，接受一个函数并返回一个新函数（通常功能增强版）：

```python
from functools import wraps

def my_decorator(fn):
    @wraps(fn)           # 保留 fn 的 __name__、__doc__ 等元信息
    def wrapper(*args, **kwargs):
        print(f"调用 {fn.__name__} 前")
        result = fn(*args, **kwargs)
        print(f"调用 {fn.__name__} 后")
        return result
    return wrapper

@my_decorator
def greet(name):
    """打招呼。"""
    print(f"Hello, {name}!")
```

`@my_decorator` 等价于 `greet = my_decorator(greet)`。

**为什么要用 `@wraps(fn)`？**
没有它，`greet.__name__` 会变成 `"wrapper"`，`greet.__doc__` 也会消失，调试时非常困惑。

---

### 上下文管理器（Context Manager）

`with` 语句保证资源被正确释放（即使中途报错）：

```python
with open("file.txt") as f:
    data = f.read()
# 此处 f 已被自动关闭
```

用 `@contextmanager` + `yield` 可以轻松实现自己的上下文管理器：

```python
from contextlib import contextmanager

@contextmanager
def timer(label):
    import time
    t0 = time.time()
    try:
        yield          # "进入 with 块"，把控制权交给调用者
    finally:
        print(f"{label}: {time.time() - t0:.3f}s")

with timer("loop"):
    total = sum(range(1_000_000))
```

执行流程：`yield` 之前的代码在进入 `with` 时运行；`yield` 之后（`finally` 里）在退出 `with` 时运行，无论是否抛出异常。

---

### 生成器（Generator）与惰性求值（Laziness）

生成器是用 `yield` 的函数，每次调用 `next()` 才计算并返回下一个值，**不提前生成所有数据**：

```python
def count_up(n):
    i = 0
    while i < n:
        yield i
        i += 1

gen = count_up(1_000_000)  # 瞬间返回，不占用 100 万个元素的内存
next(gen)                   # 0
next(gen)                   # 1
```

**好处**：处理无限序列或大文件时，不需要把所有数据一次性加载进内存。

生成器可以用 `for` 循环迭代：

```python
for x in count_up(5):
    print(x)
```


## 示例 Worked Example

下面三个示例分别对应 `w1-decorators` 和 `w1-generators` 练习包里的核心模式。


In [ ]:
# ── 示例 1：memoize 装饰器（w1-decorators）─────────────────────────────────
from __future__ import annotations
from functools import wraps


def memoize(fn):
    """缓存函数结果的装饰器（按位置参数缓存）。"""
    cache: dict = {}

    @wraps(fn)
    def wrapper(*args):
        if args not in cache:
            cache[args] = fn(*args)
        return cache[args]
    return wrapper


@memoize
def fib(n: int) -> int:
    """递归 Fibonacci（无缓存版本会非常慢）。"""
    if n <= 1:
        return n
    return fib(n - 1) + fib(n - 2)


# 验证
print([fib(i) for i in range(10)])   # [0, 1, 1, 2, 3, 5, 8, 13, 21, 34]
print(f"fib.__name__ = {fib.__name__}")  # 'fib'（@wraps 保留了名字）


In [ ]:
# ── 示例 2：tag 上下文管理器（w1-decorators）────────────────────────────────
from contextlib import contextmanager


@contextmanager
def tag(name: str, log: list):
    """记录进入/退出事件的上下文管理器。"""
    log.append(f"start:{name}")
    try:
        yield
    finally:
        log.append(f"end:{name}")


# 验证
events = []
with tag("outer", events):
    events.append("inside-outer")
    with tag("inner", events):
        events.append("inside-inner")

print("事件序列：", events)
# ['start:outer', 'inside-outer', 'start:inner', 'inside-inner', 'end:inner', 'end:outer']


In [ ]:
# ── 示例 3：moving_sum 生成器（w1-generators）───────────────────────────────
from __future__ import annotations
from typing import Iterable, Iterator


def moving_sum(xs: Iterable[float], k: int) -> Iterator[float]:
    """滑动窗口求和生成器：每次产出最新 k 个元素的和。"""
    window: list[float] = []
    for x in xs:
        window.append(x)
        if len(window) == k:
            yield sum(window)
            window.pop(0)   # 移除最旧的元素


def take(it: Iterator, n: int) -> list:
    """从迭代器中取前 n 个值。"""
    out = []
    for i, v in enumerate(it):
        if i >= n:
            break
        out.append(v)
    return out


# 验证
data = [1.0, 2.0, 3.0, 4.0, 5.0]
result = list(moving_sum(data, k=3))
print(f"moving_sum({data}, k=3) = {result}")   # [6.0, 9.0, 12.0]

# 惰性：生成器不提前计算
gen = moving_sum(range(1_000_000), k=5)
print(f"前 3 个：{take(gen, 3)}")


## 动手 Exercise

在下面的 cell 中，**实现 `@count_calls` 装饰器**：

- 被装饰的函数每次被调用时，计数器加 1
- 提供一个方法/属性 `fn.call_count` 访问调用次数
- 用 `@wraps` 保留原函数元信息

```python
@count_calls
def add(a, b):
    return a + b

add(1, 2)
add(3, 4)
print(add.call_count)  # 应该输出 2
```

> 提示：可以把计数器附加到 `wrapper` 函数对象本身（`wrapper.call_count = 0`）。


In [ ]:
from functools import wraps


def count_calls(fn):
    """计数调用次数的装饰器。"""
    # TODO: 实现
    @wraps(fn)
    def wrapper(*args, **kwargs):
        ...
    wrapper.call_count = 0
    return wrapper


In [ ]:
# 验证
try:
    @count_calls
    def add(a, b):
        return a + b

    add(1, 2)
    add(3, 4)
    add(5, 6)
    print("call_count == 3 ?", add.call_count == 3)
    print("返回值正确 ?", add(10, 20) == 30)
    print("call_count == 4 ?", add.call_count == 4)
    print("__name__ 保留 ?", add.__name__ == "add")
except Exception as e:
    print("出错：", e)


## 延伸阅读 Further Reading

1. **Python 官方文档 — functools**：<https://docs.python.org/3/library/functools.html>
2. **Python 官方文档 — contextlib**：<https://docs.python.org/3/library/contextlib.html>
3. **PEP 343 — `with` 语句**：<https://peps.python.org/pep-0343/>
4. **PEP 255 — 简单生成器**：<https://peps.python.org/pep-0255/>
5. **Real Python: Primer on Python Decorators**（可 Google 标题）——有非常清晰的逐步推导。
6. **《流畅的 Python》（Fluent Python）第 7、9、14 章**：闭包、装饰器、迭代器/生成器的深度讲解，中文版可找到。


## 关联练习 Related Assignments

本课对应两个练习包：

1. **`assessments/autograder/w1-decorators`**（`memoize` + `tag` 上下文管理器）
2. **`assessments/autograder/w1-generators`**（`moving_sum` + `take` 生成器）

**运行自动评分器**：

```bash
cd assessments/autograder/w1-decorators
python -m pytest test_hidden.py -v

cd assessments/autograder/w1-generators
python -m pytest test_hidden.py -v
```

> 目标通过率 ≥ 70%（`threshold_pass: 0.7`）
> 能力标签：**SH1**
